## 1. Imports & Load Raw Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [3]:
df = pd.read_csv('dataset/Customers.csv')
df.head(5)

,CustomerID,Gender,Age,Annual Income ($),Spending Score (1-100),Profession,Work Experience,Family Size
0,1,Male,19,15000,39,Healthcare,1,4
1,2,Male,21,35000,81,Engineer,3,3
2,3,Female,20,86000,6,Engineer,1,1
3,4,Female,23,59000,77,Lawyer,0,2
4,5,Female,31,38000,40,Entertainment,2,6


## 2. Handle Missing / Invalid Values

In [5]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 35
Duplicates: 0
Missing Feature:
['Profession']


In [6]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    
    if missing_percentage > 5.0:
        df = df.drop(columns=col)
    else:
        if pd.api.types.is_timedelta64_dtype(df[col]):
            median_timedelta = df[col].median()
            df[col] = df[col].fillna(median_timedelta)
        
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            median_date = df[col].median()
            df[col] = df[col].fillna(median_date)

        elif df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')


Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [8]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [10]:
feature_numerik = df.select_dtypes(include=np.number).columns
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 5
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering

### 5.1 Drop Feature ID

In [13]:
df = df.drop(columns=['CustomerID'])